# ICS 604: APPLIED DATA SCIENCE

## Parameter Estimation: Bootstrap Confidence Interval
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Bootstrapping

Bootstrapping is a resampling technique that consists of sampling _with replacement_ from an observed dataset. Instead of drawing new samples from the true (and usually unknown) population, we repeatedly draw new samples from the data we already have.

The basic idea is that *inference about a population from a sample can be approximated by resampling from that sample and performing inference on the resampled datasets*. In other words, we treat the observed sample as if it were a proxy for the population and simulate repeated sampling from it.

### How Bootstrapping Is Used

Bootstrapping allows us to assess the **extent of sampling variability** — that is, how much a statistic would vary if we were able to repeatedly collect new samples from the population.

It is commonly used to:

- Approximate the *sampling distribution* of a statistic (e.g., mean, median, variance)
- Estimate *standard errors*
- Construct *confidence intervals*
- Assess stability of model estimates

The procedure typically involves:
1. Drawing many resamples (with replacement) from the original dataset.
2. Computing the statistic of interest for each resample.
3. Examining the distribution of those computed statistics.

Importantly, in bootstrap resampling, the “population” is effectively the original sample itself. By repeatedly resampling from it, we approximate the variability we would expect if we could repeatedly sample from the true underlying population.

### Using Bootstrap to Estimate the Population Mean

In this example, we generate a sample of size 100 from a normal distribution with mean $\mu=50$ and standard deviation $\sigma=10$. The data are simulated as follows:

```python
np.random.seed(22)
data = np.random.normal(50, 10, 100)
```

Using this sample, we aim to apply the bootstrap procedure to estimate the population mean and assess its variability. By repeatedly resampling with replacement from the observed data and computing the mean for each resample, we can approximate the sampling distribution of the mean.

* Are 100 records sufficient to obtain a good measure of the population mean using bootstrap?
* What is the population’s range of possible means that can be estimated from samples of this size?

In [ ]:
# Given the following

np.random.seed(22)

data = np.random.normal(50, 10, 100)
data.mean()

In [ ]:
data

In [ ]:
# One bootstrap resample - Number of distinct values

len(set(np.random.choice(data, 100)))

In [ ]:
bootstrap_means = []

for i in range(10_000):
    returns_data_100_bootstrap = np.random.choice(data, 100)
    bootstrap_mean = returns_data_100_bootstrap.mean()
    bootstrap_means.append(bootstrap_mean)

In [ ]:
bootstrap_means[:100]

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(bootstrap_means, density=True, edgecolor='black', linewidth=1.2, alpha=0.5)
plt.show()

In [ ]:
np.percentile(bootstrap_means, (2.5, 97.5))

### Interpreting the Bootstrap Values

The bootstrap distribution provides insight into the extent of sampling variability in our estimate of the mean. By repeatedly resampling from the observed dataset and computing the mean each time, we obtain an empirical distribution of possible mean values that could have arisen from samples of this size.

From this bootstrap distribution, we construct a 95% confidence interval for the mean. Values that fall within this interval are not considered rare or extreme under repeated sampling. In other words, we cannot reasonably dismiss any value within the 95% confidence interval as implausible for the true population mean based on the observed data.

In this example, even with only 100 samples, the bootstrap procedure yields a confidence interval of approximately $[47.49,51.36]$. This interval contains the true population mean, illustrating how bootstrap resampling can provide a reliable estimate of uncertainty even with a moderately sized sample.

### How Confident Are We in the Bootstrap Confidence Interval?

We observed that the estimated 95% confidence interval captures the population mean. But was that simply a fluke?

To understand how frequently such an interval truly contains the population parameter, we must repeat the entire procedure many times. The idea is to evaluate the **long-run performance** of the bootstrap confidence interval.

Specifically, we repeat the following process multiple times (for example, 100 repetitions):

  * Generate a new data sample of size 100 from the population.
  * Generate 10,000 bootstrap resamples from that data sample.
  * Compute the 95% confidence interval for the mean
  
At the end of this experiment, we will have 100 different confidence intervals. We then count how many of those intervals contain the true population mean.

How many of these 95% confidence intervals should include the true mean?

In [ ]:
def compute_conf_interval(data, nb_bootstrap_iters = 10_000):
    bootstrap_means = []
    for i in range(nb_bootstrap_iters):
        bootstrap_sample = np.random.choice(data, 100, replace=True)
        bootstrap_means.append(bootstrap_sample.mean())
    return np.percentile(bootstrap_means, (2.5, 97.5))
    
lower_bound = []
upper_bound = []
for i in range(100):
    data = np.random.normal(50, 10, 100)
    conf_interval = compute_conf_interval(data)
    lower_bound.append(conf_interval[0])
    upper_bound.append(conf_interval[1])

In [ ]:
conf_ints_95 = pd.DataFrame({"lower": lower_bound, "upper": upper_bound})
conf_ints_95.head()

In [ ]:
conf_ints_95.shape

In [ ]:
sum((conf_ints_95["lower"] < 50) & (conf_ints_95["upper"] > 50))

In [ ]:
plt.figure(figsize=(10, 6))
plt.vlines(50, 0, 102, color="#e2a829", linewidth=4)

for i in range(100):
    c = "blue"
    if lower_bound[i] > 50 or upper_bound[i] < 50:
        c = "red"
    plt.hlines(i, lower_bound[i], upper_bound[i], color=c, alpha=0.85, linewidth=0.75)

If an interval fails to cover the parameter, it is considered a failure. Even with a correct method, some failures are expected. Any inference procedure based on sampling carries the possibility of error.

With a 95% confidence interval, we expect to be wrong approximately 5% of the time. This statement refers to the **statistical expectation over a long run of repeated experiments**. If we were to construct confidence intervals an extremely large number of times under identical conditions, about 95% of those intervals would contain the true parameter, while roughly 5% would not.

The strength of sampling-based methods lies in this quantification: not only do they provide estimates, but they also allow us to measure how often those estimates are likely to be incorrect.

### Misinterpretations and Misunderstandings of Confidence Intervals

A common misunderstanding is to say that a 95% confidence interval means there is a 95% probability that the specific calculated interval contains the true parameter. This interpretation is incorrect. The true parameter is fixed, and once the interval is computed from the observed data, the interval either contains the parameter or it does not. There is no probability attached to that specific realized interval.

The correct interpretation is rooted in repeated sampling. A 95% confidence level means that if we were to repeatedly draw samples from the population and construct a confidence interval from each sample using the same procedure, approximately 95% of those intervals would contain the true parameter value. About 5% would fail to do so.

This distinction is subtle but fundamental. The randomness lies in the sampling process and in the interval construction procedure, not in the parameter itself. Once a specific interval has been calculated, it is fixed, and the true parameter either falls inside it or it does not.

### The Bootstrap Confidence Interval

The interval of estimates obtained through the bootstrap procedure is called a **95% confidence interval** for the parameter of interest. This interval represents a range of plausible values for the true population parameter based on the observed data and the resampling process.

The value 95% is referred to as the **confidence level** of the interval. It reflects the long-run proportion of similarly constructed intervals that would contain the true parameter if the sampling and bootstrap procedure were repeated many times under the same conditions.

When we say that we are **95% confident**, we mean that the method used to construct the interval has a 95% success rate in capturing the true parameter over repeated samples. The confidence refers to the reliability of the procedure, not to the probability that the specific computed interval contains the parameter.

### Why is the Bootstrap a Good Idea? 

The bootstrap works well because of a principle of **similarity by transitivity**. By the law of averages, the distribution of a sufficiently large sample is likely to resemble the underlying population distribution. Consequently, when we draw resamples from the original sample, the distributions of these resamples are likely to resemble the original sample, and therefore also approximate the population distribution.

In essence, we **treat the original sample as if it were the entire population**. Bootstrap resampling involves drawing observations from this sample **with replacement**, ensuring that each resample can include repeated observations. Each bootstrap sample is typically the same size as the original dataset, which avoids introducing discrepancies due to differences in sample size alone.

By repeatedly resampling and computing the statistic of interest, the bootstrap provides a practical, data-driven way to estimate sampling variability and construct confidence intervals **without needing strong parametric assumptions**.

<img src="https://www.dropbox.com/scl/fi/yg9iiljcu5iozzanx874s/bootstrap_pic.png?rlkey=9bmy17mb0hru19rgzudzsod9r&st=vmdjjhau&dl=1" alt="drawing" style="width:800px"/>

### Care in Using the Bootstrap

The bootstrap is an elegant and powerful method for assessing the accuracy of an estimate, particularly for estimating the variability of a statistic. Unlike traditional parametric approaches, the bootstrap uses a resampling-based method to estimate standard errors and construct confidence intervals, relying directly on the observed data rather than assuming a specific population model.

Before applying the bootstrap, there are a few important considerations:

- **Start with a sufficiently large random sample.** The reliability of bootstrap estimates improves as the size of the original sample increases. The Law of Large Numbers suggests that larger samples are more likely to resemble the population distribution, which is critical for the validity of resampling.

- **Replicate the resampling procedure many times.** To approximate the sampling distribution of a statistic accurately, it is advisable to perform a large number of bootstrap iterations — typically on the order of 10,000 resamples.

- **Method suitability.** The bootstrap percentile method works particularly well for estimating parameters such as the population mean or median when based on a large, random sample. By constructing intervals from the percentiles of the bootstrap distribution, we obtain practical and interpretable estimates of uncertainty.

Overall, careful attention to sample size, the number of resamples, and the choice of statistic ensures that bootstrap methods produce reliable and meaningful inference.

### When To Not Use the Bootstrap

While the bootstrap is a powerful tool, it is not appropriate in all situations. There are specific scenarios where its assumptions and methodology may lead to misleading results:

- **Estimating extreme values or rare-event parameters:** The bootstrap performs poorly when the goal is to estimate the population minimum or maximum, very low or very high percentiles, or other statistics heavily influenced by rare elements. In such cases, resampling from the original sample may fail to capture these extremes adequately.

- **Non-normal or irregular statistic distributions:** If the probability distribution of the statistic is far from bell-shaped, bootstrap estimates may be inaccurate. While the method can tolerate moderate skewness, extreme departures from a roughly symmetric distribution can compromise validity.

- **Very small sample sizes:** When the original dataset is extremely small (e.g., fewer than 20–25 observations), the sample may not adequately represent the population. Consequently, bootstrap resamples drawn from such a sample are unlikely to approximate the true variability of the statistic, reducing the reliability of the resulting confidence intervals.

In short, the bootstrap works best with moderate-to-large, representative random samples and for statistics that are not overly sensitive to rare events or extreme values.